# AFDM 第三步：整数/分数多普勒，已知信道

独立 NumPy 软件实验：上传后在新 Python 内核中运行全部代码即可，不需要先执行 02 或 03。
不调用 PYNQ、FPGA、ADC/DAC 或 DMA。

物理模型采用 [AFDM 原论文](https://arxiv.org/html/2204.12798v2) 式 (20)～(25)：
`r[n] = sum_i h_i * exp(-j*2*pi*nu_i*n/N) * s[n-delay_i] + w[n]`。
nu 按子载波间隔归一化；nu=1 表示一个子载波间隔，不直接表示 1 Hz。
整数用例为 `[0,1,-2]`，分数用例为 `[0.25,1.4,-1.7]`。

本轮包括时域延迟与逐路径多普勒、CPP、有效信道矩阵、ZF/LMMSE、已知比特误码统计。
接收端仍使用真实信道、完美帧位置，尚未从导频估计信道，没有同步、射频或 FPGA 验证。
密集矩阵检测器仅作小规模正确性参考。单个固定信道的结果不能证明普遍性能或分集优势。


In [ ]:
import numpy as np

SEED = 2026111704
N = 64
BLOCKS = 256
L = 16
ALPHA_MAX = 2
c1 = (2*ALPHA_MAX+1)/(2*N)
c2 = np.sqrt(2.0)/(N*N)
general_c1 = 0.37/N
delays = np.array([0,2,5], dtype=int)
gains = np.array([1.0,0.55*np.exp(0.4j),0.25*np.exp(-0.8j)])
gains /= np.linalg.norm(gains)
CASES = {'static': np.array([0.,0.,0.]),
         'integer': np.array([0.,1.,-2.]),
         'fractional': np.array([0.25,1.4,-1.7])}
SNR_DB = [0,10,20]
rng = np.random.default_rng(SEED)
passed = []

def mark(number, description):
    assert number == len(passed)+1
    passed.append(number)
    print('PASS %d: %s' % (number, description))

print('NumPy:', np.__version__, '; N=',N,'; L=',L,'; blocks=',BLOCKS)
print('c1=',c1,'; c2=',c2,'; seed=',SEED)
print('Doppler convention: exp(-j*2*pi*nu*n/N)')
def idaft(symbols, c1, c2):
    """输入最后一维为 DAFT 符号，输出单位化的时域波形。"""
    symbols = np.asarray(symbols, dtype=np.complex128)
    n = np.arange(symbols.shape[-1])
    return np.exp(2j*np.pi*c1*n*n) * np.fft.ifft(
        symbols * np.exp(2j*np.pi*c2*n*n), axis=-1, norm='ortho')

def daft(samples, c1, c2):
    """输入时域采样，输出 DAFT 域符号。"""
    samples = np.asarray(samples, dtype=np.complex128)
    n = np.arange(samples.shape[-1])
    return np.exp(-2j*np.pi*c2*n*n) * np.fft.fft(
        samples * np.exp(-2j*np.pi*c1*n*n), axis=-1, norm='ortho')

def qpsk_map(bits):
    bits = np.asarray(bits)
    return ((1.0-2.0*bits[..., 0]) + 1j*(1.0-2.0*bits[..., 1])) / np.sqrt(2.0)

def qpsk_bits(symbols):
    return np.stack([symbols.real < 0, symbols.imag < 0], axis=-1).astype(np.uint8)

def add_cpp(samples, length, chirp_c1):
    samples = np.asarray(samples, dtype=np.complex128)
    size = samples.shape[-1]
    if not isinstance(length, (int, np.integer)) or not 0 <= length <= size:
        raise ValueError('CPP length must be an integer between 0 and N')
    if length == 0:
        return samples.copy()
    negative_n = np.arange(-length, 0)
    phase = np.exp(-2j*np.pi*chirp_c1*(size*size + 2*size*negative_n))
    return np.concatenate([samples[..., -length:]*phase, samples], axis=-1)

def time_channel_matrix(size, chirp_c1, path_delays=delays, path_gains=gains):
    """用索引和前缀边界相位独立构造 H，与线性卷积输出对照。"""
    if max(path_delays) >= size:
        raise ValueError('Delays must be less than N')
    matrix = np.zeros((size, size), dtype=np.complex128)
    n = np.arange(size)
    for delay, gain in zip(path_delays, path_gains):
        k = n-delay
        phase = np.ones(size, dtype=np.complex128)
        mask = k < 0
        phase[mask] = np.exp(-2j*np.pi*chirp_c1*(size*size+2*size*k[mask]))
        matrix[n, k % size] += gain*phase
    return matrix

def daft_matrix(size, chirp_c1, chirp_c2):
    m = np.arange(size)[:, None]
    n = np.arange(size)[None, :]
    return np.exp(-2j*np.pi*(chirp_c2*m*m+m*n/size+chirp_c1*n*n))/np.sqrt(size)


In [ ]:
def linear_doppler_channel(signal, path_delays, path_gains, path_nu, size, time_origin=0):
    """延迟后按输出的真实时间加逐路径多普勒，保持连续块的相位连续。"""
    signal = np.asarray(signal, dtype=np.complex128)
    d = np.asarray(path_delays)
    nu = np.asarray(path_nu)
    if d.ndim != 1 or not len(d) or len(d) != len(path_gains) or len(d) != len(nu):
        raise ValueError('Channel path arrays must have matching lengths')
    if not np.issubdtype(d.dtype, np.integer) or np.any(d < 0) or not np.all(np.isfinite(nu)):
        raise ValueError('Nonnegative integer delays and finite Doppler required')
    received = np.zeros(signal.shape[:-1]+(signal.shape[-1]+int(max(d)),), dtype=np.complex128)
    t = np.arange(received.shape[-1])+time_origin
    for delay, gain, doppler in zip(d, path_gains, nu):
        length = signal.shape[-1]
        phase = np.exp(-2j*np.pi*doppler*t[delay:delay+length]/size)
        received[...,delay:delay+length] += gain*signal*phase
    return received

def receive_doppler_block(samples, length, chirp_c1, path_nu, path_delays=delays,
                          path_gains=gains, time_start=0):
    if length < max(path_delays):
        raise ValueError('Prefix is shorter than maximum delay')
    full = linear_doppler_channel(add_cpp(samples,length,chirp_c1),path_delays,path_gains,
                                 path_nu,samples.shape[-1],time_start-length)
    return full[...,length:length+samples.shape[-1]]

def doppler_channel_matrix(size, chirp_c1, path_nu, path_delays=delays,
                           path_gains=gains, time_start=0):
    """用逐行的输出时间、循环索引与 CPP 边界相位，独立构造 H。"""
    if max(path_delays) >= size:
        raise ValueError('Delays must be less than N')
    H = np.zeros((size,size),dtype=np.complex128)
    n = np.arange(size)
    for delay,gain,nu in zip(path_delays,path_gains,path_nu):
        k = n-delay
        prefix_phase = np.ones(size,dtype=np.complex128)
        mask = k < 0
        prefix_phase[mask] = np.exp(-2j*np.pi*chirp_c1*(size*size+2*size*k[mask]))
        phase = np.exp(-2j*np.pi*nu*(n+time_start)/size)
        H[n,k % size] += gain*phase*prefix_phase
    return H


## 模型正确性：物理通路、解析矩阵与连续相位

In [ ]:
bits = rng.integers(0,2,size=(BLOCKS,N,2),dtype=np.uint8)
symbols = qpsk_map(bits)
modes = [('AFDM',c1,c2),('OFDM',0.,0.)]
for chirp in [0.,c1,general_c1]:
    H0 = doppler_channel_matrix(N,chirp,CASES['static'])
    assert np.max(np.abs(H0-time_channel_matrix(N,chirp))) < 1e-12
mark(1,'零多普勒时回到上一阶段静态信道矩阵')

# 独立单音检查频移的符号和归一化：nu=+1 应移到 FFT 的 -1 栅格。
tone = np.ones(N,dtype=np.complex128)
one_path = dict(path_delays=np.array([0],dtype=int),path_gains=np.array([1.+0j]))
shifted = receive_doppler_block(tone,L,0.,np.array([1.]),**one_path)
assert np.max(np.abs(shifted-np.exp(-2j*np.pi*np.arange(N)/N))) < 1e-12
assert np.argmax(np.abs(np.fft.fft(shifted))) == N-1
mark(2,'单音频移方向与 nu/N 的归一化正确')

channel_errors = []
for case,nu in CASES.items():
    for chirp in [0.,c1,general_c1]:
        wave = idaft(symbols[:4],chirp,c2)
        received = receive_doppler_block(wave,L,chirp,nu)
        H = doppler_channel_matrix(N,chirp,nu)
        error = float(np.max(np.abs(received-wave @ H.T)))
        A = daft_matrix(N,chirp,c2)
        effective_error = float(np.max(np.abs(daft(received,chirp,c2)-symbols[:4] @ (A @ H @ A.conj().T).T)))
        assert error < 1e-11 and effective_error < 1e-10
        channel_errors.append(dict(case=case,c1=chirp,time_error=error,effective_error=effective_error))
mark(3,'静态/整数/分数多普勒中，时域通路与 H、H_eff 矩阵一致')

# 用论文式 (30) 单独核对一条整数多普勒路径的移位及相位。
A = daft_matrix(N,c1,c2)
path_delay, path_nu = 2, 1
Hp = doppler_channel_matrix(N,c1,np.array([float(path_nu)]),
     path_delays=np.array([path_delay]),path_gains=np.array([1.+0j]))
Hp_eff = A @ Hp @ A.conj().T
p = np.arange(N)
q = (p+path_nu+int(round(2*N*c1))*path_delay) % N
analytic = np.zeros((N,N),dtype=np.complex128)
analytic[p,q] = np.exp(2j*np.pi/N*(N*c1*path_delay**2-q*path_delay+N*c2*(q*q-p*p)))
integer_formula_error = float(np.max(np.abs(Hp_eff-analytic)))
assert integer_formula_error < 1e-10
assert np.all(np.sum(np.abs(Hp_eff)>1e-10,axis=1)==1)
mark(4,'整数多普勒单路径符合解析式，每行一个非零位置')

Hf = doppler_channel_matrix(N,c1,np.array([1.4]),
     path_delays=np.array([path_delay]),path_gains=np.array([1.+0j]))
Hf_eff = A @ Hf @ A.conj().T
assert np.all(np.sum(np.abs(Hf_eff)>1e-10,axis=1)>1)
assert np.max(np.abs(Hf_eff.conj().T @ Hf_eff-np.eye(N))) < 1e-10
mark(5,'分数多普勒单路径出现跨栅格扩散，单路径能量仍守恒')

# 连续两块不能在边界随意将 Doppler 相位复位为零。
nu = CASES['fractional']
for chirp in [c1,general_c1]:
    wave_pair = idaft(symbols[:2],chirp,c2)
    framed = add_cpp(wave_pair,L,chirp).reshape(-1)
    stream = linear_doppler_channel(framed,delays,gains,nu,N,time_origin=-L)
    offset = N+2*L
    second = stream[offset:offset+N]
    H2 = doppler_channel_matrix(N,chirp,nu,time_start=N+L)
    assert np.max(np.abs(second-H2 @ wave_pair[1])) < 1e-11
    Hwrong = doppler_channel_matrix(N,chirp,nu,time_start=0)
    assert np.max(np.abs(second-Hwrong @ wave_pair[1])) > 0.1
mark(6,'连续块相位测试通过，第二块使用正确的绝对时间')


## 无噪声已知信道：三个场景、两种波形

In [ ]:
systems = {}
clean_rows = []
print('Case          Mode   errors/bits      max_symbol_error')
for case,nu in CASES.items():
    for name,mode_c1,mode_c2 in modes:
        wave = idaft(symbols,mode_c1,mode_c2)
        received = receive_doppler_block(wave,L,mode_c1,nu)
        A = daft_matrix(N,mode_c1,mode_c2)
        H_eff = A @ doppler_channel_matrix(N,mode_c1,nu) @ A.conj().T
        Y = daft(received,mode_c1,mode_c2)
        recovered = np.linalg.solve(H_eff,Y.T).T
        errors = int(np.count_nonzero(qpsk_bits(recovered)!=bits))
        max_error = float(np.max(np.abs(recovered-symbols)))
        assert errors == 0 and max_error < 1e-10
        clean_rows.append(dict(case=case,mode=name,errors=errors,bits=int(bits.size),
                               max_symbol_error=max_error,condition_number=float(np.linalg.cond(H_eff))))
        systems[(case,name)] = (mode_c1,mode_c2,received,H_eff)
        print('%-13s %-5s %5d/%-8d %.3e' % (case,name,errors,bits.size,max_error))
mark(7,'三个场景的 AFDM/OFDM 均可用真实信道 ZF 恢复全部已知比特')


## 带噪声的完整已知信道 LMMSE 检测
SNR 为发射 Es/N0，不计前缀开销。没有从接收数据估计信道，也不假设所有路径具有同一个可一次补偿的频偏。

In [ ]:
# 同一场景中两波形使用相同比特、功率、CPP长度和噪声样本。
# 两者均使用完整已知信道 LMMSE；不把仅适合静态信道的 OFDM 单抽头均衡器用于对照。
# 各块按相同的有用采样起点独立仿真；连续块相位另由 PASS 6 检查。
noisy_rows = []
print('Case          SNR  AFDM_errors/bits  OFDM_errors/bits')
for case in CASES:
    case_rows = []
    for snr_db in SNR_DB:
        variance = 10.0**(-snr_db/10.0)
        w = np.sqrt(variance/2)*(rng.standard_normal((BLOCKS,N))+1j*rng.standard_normal((BLOCKS,N)))
        row = dict(case=case,snr_db=snr_db,bits=int(bits.size),noise_variance=variance)
        for name,_,_ in modes:
            mode_c1,mode_c2,received,H_eff = systems[(case,name)]
            Y = daft(received+w,mode_c1,mode_c2)
            W = np.linalg.solve(H_eff.conj().T @ H_eff+variance*np.eye(N),H_eff.conj().T)
            estimate = Y @ W.T
            errors = int(np.count_nonzero(qpsk_bits(estimate)!=bits))
            row[name] = dict(errors=errors,ber=errors/bits.size,
                             mse=float(np.mean(np.abs(estimate-symbols)**2)))
        noisy_rows.append(row)
        case_rows.append(row)
        print('%-13s %3d %5d/%-8d %5d/%-8d' %
              (case,snr_db,row['AFDM']['errors'],bits.size,row['OFDM']['errors'],bits.size))
    for name,_,_ in modes:
        assert case_rows[-1][name]['mse'] < case_rows[0][name]['mse']
        assert case_rows[-1][name]['errors'] < case_rows[0][name]['errors']
mark(8,'三个场景中提高 SNR 后已知信道检测误差下降')
print('这些固定信道、有限比特统计不用于证明普遍性能或分集优势。')


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import uuid

assert passed == list(range(1,9))
assert len(clean_rows)==6 and all(row['errors']==0 for row in clean_rows)
result_dir = Path.cwd()/('afdm_doppler_results_'+uuid.uuid4().hex[:8])
result_dir.mkdir(exist_ok=False)
summary = dict(scope='software-only; known channel; perfect timing; no estimation or RF',
    host_clock=datetime.now().isoformat(),numpy_version=np.__version__,seed=SEED,n=N,blocks=BLOCKS,
    cpp_length=L,c1=c1,c2=float(c2),passed_checks=passed,
    doppler_sign='exp(-j*2*pi*nu*n/N)',doppler_units='subcarrier spacing, not Hz',
    noise_snr_definition='transmit Es/N0; prefix energy overhead excluded',
    paths=[dict(delay=int(d),gain_real=float(h.real),gain_imag=float(h.imag)) for d,h in zip(delays,gains)],
    cases={k:v.tolist() for k,v in CASES.items()},channel_matrix_checks=channel_errors,
    integer_formula_error=integer_formula_error,clean_results=clean_rows,noisy_results=noisy_rows)
(result_dir/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
np.savez(result_dir/'reference_vectors.npz',bits=bits[0],symbols=symbols[0],
    tx_afdm=idaft(symbols[0],c1,c2),delays=delays,gains=gains,
    fractional_nu=CASES['fractional'],c1=c1,c2=c2,cpp_length=L)
print('验证汇总：PASS 8/8；无噪声 6/6 组均为 0 / %d 个错误。' % bits.size)
print('结果保存位置：',result_dir)
print('下一步：导频与信道估计；同步、射频和 FPGA 实现仍待完成。')
